# 🎓 Prédiction de Réussite Scolaire — Analyse complète

Ce notebook réalise **deux analyses complémentaires** à partir des données socio-scolaires des étudiants :

| Partie | Tâche | Cible | Modèles |
|--------|-------|-------|---------|
| **1 — Classification** | Prédit si l'élève **passe l'année** | `passe_annee` (0/1) | MLP 1, MLP 2, SVM, Random Forest |
| **2 — Régression** | Prédit la **note finale G3** (0–20) | `G3` (entier) | MLP 1, MLP 2, SVR, Random Forest |

**Architecture des MLP :**
- MLP 1 : `n_features → 10 neurones → 1 sortie`
- MLP 2 : `n_features → 5 neurones → 1 sortie`

**Données :** `student-por.csv` — 649 élèves, 33 colonnes (notes + caractéristiques démographiques/comportementales)


---
## 📦 1. Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, accuracy_score, classification_report,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.svm import SVC, SVR
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

tf.random.set_seed(42)
np.random.seed(42)

print(f"✅ Imports OK — TensorFlow {tf.__version__}")

---
## 📂 2. Chargement & encodage des données

In [ ]:
# ── Dictionnaire d'encodage des colonnes catégorielles ──────────────────────
ENCODAGE = {
    "school":    {"GP": 0, "MS": 1},
    "sex":       {"F": 0, "M": 1},
    "address":   {"U": 0, "R": 1},
    "famsize":   {"GT3": 0, "LE3": 1},
    "Pstatus":   {"T": 0, "A": 1},
    "Mjob":      {"teacher": 0, "health": 1, "services": 2, "at_home": 3, "other": 4},
    "Fjob":      {"teacher": 0, "health": 1, "services": 2, "at_home": 3, "other": 4},
    "reason":    {"home": 0, "reputation": 1, "course": 2, "other": 3},
    "guardian":  {"father": 0, "mother": 1, "other": 2},
    "schoolsup": {"yes": 0, "no": 1},
    "famsup":    {"yes": 0, "no": 1},
    "paid":      {"yes": 0, "no": 1},
    "activities":{"yes": 0, "no": 1},
    "nursery":   {"yes": 0, "no": 1},
    "higher":    {"yes": 0, "no": 1},
    "internet":  {"yes": 0, "no": 1},
    "romantic":  {"yes": 0, "no": 1},
}

def charger_et_encoder(chemin):
    """Charge le CSV et encode toutes les colonnes catégorielles."""
    data = pd.read_csv(chemin, sep=",", encoding="utf8")
    print(f"✅ Fichier chargé : {data.shape[0]} lignes × {data.shape[1]} colonnes")
    for col, mapping in ENCODAGE.items():
        if col in data.columns:
            data[col] = data[col].map(mapping)
    for col in ["G1", "G2", "G3"]:
        data[col] = data[col].astype(int)
    print(f"   Valeurs manquantes : {data.isnull().sum().sum()}")
    return data

data_enc = charger_et_encoder("Data/student-por.csv")
display(data_enc.head())

---
# 🔵 PARTIE 1 — Classification : L'élève passe-t-il l'année ?

**Principe :** On utilise uniquement les données **démographiques et comportementales** (sans les notes G1/G2/G3) pour prédire si la moyenne annuelle est ≥ 10/20.

> ⚠️ Les données sont **déséquilibrées** : une majorité d'élèves réussissent. L'accuracy seule est donc insuffisante — on analysera aussi précision, rappel et F1-score.


---
## 🔧 3. Prétraitement — Classification

In [ ]:
# ── Construction du jeu de données de classification ────────────────────────
df_clf = data_enc.copy()

# Cible : passage de l'année (moyenne des 3 trimestres >= 10/20)
df_clf["Moyenne"] = (df_clf["G1"] + df_clf["G2"] + df_clf["G3"]) / 3
df_clf["passe_annee"] = (df_clf["Moyenne"] >= 10).astype(int)

# Suppression des colonnes de notes (prédiction purement socio-démographique)
df_clf.drop(columns=["G1", "G2", "G3", "Moyenne"], inplace=True)

X_clf = df_clf.drop(columns=["passe_annee"])
y_clf = df_clf["passe_annee"]

# ── Statistiques ─────────────────────────────────────────────────────────────
counts = y_clf.value_counts().sort_index()
print(f"Features : {X_clf.shape[1]}  |  Exemples : {X_clf.shape[0]}")
print("\nDistribution de la cible :")
for k, v in counts.items():
    lbl = "Réussite" if k == 1 else "Échec"
    print(f"  {lbl} ({k}) : {v:3d}  ({v/len(y_clf)*100:.1f}%)")

# ── Graphique de distribution ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 4))
colors_bar = ["#e74c3c", "#2ecc71"]
bars = ax.bar(["Échec (0)", "Réussite (1)"],
              [counts.get(0, 0), counts.get(1, 0)],
              color=colors_bar, edgecolor="black", width=0.45)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 2, str(int(bar.get_height())),
            ha="center", fontsize=12, fontweight="bold")
ax.set_title("Distribution de la cible — Passage de l'année", fontsize=13)
ax.set_ylabel("Nombre d'élèves")
ax.set_ylim(0, max(counts.values) * 1.2)
plt.tight_layout()
plt.show()

---
## 📊 4. Matrice de corrélation — Classification

In [ ]:
# ── Matrice de corrélation complète ─────────────────────────────────────────
corr_clf = df_clf.corr()

plt.figure(figsize=(15, 13))
mask = np.triu(np.ones_like(corr_clf, dtype=bool))
sns.heatmap(corr_clf, mask=mask, annot=False, cmap="coolwarm", center=0,
            linewidths=0.3, linecolor="white",
            cbar_kws={"label": "Coefficient de Pearson", "shrink": 0.8})
plt.title("Matrice de corrélation — Données de classification", fontsize=14, pad=14)
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()

# ── Top corrélations avec la cible ───────────────────────────────────────────
print("\n🔍 Top 10 corrélations avec 'passe_annee' :")
top_clf = (corr_clf["passe_annee"]
           .drop("passe_annee")
           .sort_values(key=lambda s: s.abs(), ascending=False)
           .head(10))
for feat, val in top_clf.items():
    sign = "+" if val > 0 else "-"
    bar = "█" * int(abs(val) * 20)
    print(f"  {sign}{abs(val):.4f}  {bar:<20}  {feat}")

---
## ✂️ 5. Séparation train/test + normalisation — Classification

In [ ]:
# ── Split stratifié (conserve les proportions des classes) ──────────────────
X_tr_clf, X_te_clf, y_tr_clf, y_te_clf = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

scaler_clf = StandardScaler()
X_tr_clf_sc = scaler_clf.fit_transform(X_tr_clf)
X_te_clf_sc  = scaler_clf.transform(X_te_clf)

y_tr_clf_np = y_tr_clf.values
y_te_clf_np = y_te_clf.values

print(f"Train : {X_tr_clf_sc.shape}  |  Test : {X_te_clf_sc.shape}")
print(f"Distribution train — Échec : {(y_tr_clf_np==0).sum()}  Réussite : {(y_tr_clf_np==1).sum()}")
print(f"Distribution test  — Échec : {(y_te_clf_np==0).sum()}  Réussite : {(y_te_clf_np==1).sum()}")

---
## 🛠️ 6. Fonctions d'évaluation — Classification

In [ ]:
def eval_clf(y_true, y_pred, nom):
    """
    Évalue un modèle de classification :
    - Accuracy (avec mise en garde sur le déséquilibre)
    - Rapport de classification complet (précision, rappel, F1)
    - Matrice de confusion annotée
    """
    acc = accuracy_score(y_true, y_pred)
    cm  = confusion_matrix(y_true, y_pred)

    print(f"\n{'='*58}")
    print(f"  {nom}")
    print(f"  Accuracy : {acc*100:.2f}%  ⚠ (données déséquilibrées)")
    print(f"{'='*58}")
    print(classification_report(y_true, y_pred,
                                target_names=["Échec", "Réussite"],
                                zero_division=0))

    # Matrice de confusion
    tn, fp, fn, tp = cm.ravel() if cm.shape == (2, 2) else (0, 0, 0, 0)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Prédit Échec", "Prédit Réussite"],
                yticklabels=["Réel Échec",   "Réel Réussite"],
                ax=ax, linewidths=1, linecolor="white")
    ax.set_xlabel("Valeur prédite", fontsize=11)
    ax.set_ylabel("Valeur réelle",  fontsize=11)
    ax.set_title(
        f"Matrice de Confusion — {nom}\n"
        f"VN={tn}  FP={fp}  FN={fn}  VP={tp}  |  Accuracy={acc:.4f}",
        fontsize=10
    )
    plt.tight_layout()
    plt.show()
    return acc


def plot_loss_clf(history, titre):
    """Courbes d'apprentissage Loss + Accuracy pour un MLP classifieur."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

    ax1.plot(history.history["loss"],     label="Train",      color="#2980b9")
    ax1.plot(history.history["val_loss"], label="Validation",  color="#e74c3c", ls="--")
    ax1.set_title(f"{titre} — Loss (binary cross-entropy)")
    ax1.set_xlabel("Epochs"); ax1.set_ylabel("Loss"); ax1.legend()

    ax2.plot(history.history["accuracy"],     label="Train",     color="#27ae60")
    ax2.plot(history.history["val_accuracy"], label="Validation", color="#e67e22", ls="--")
    ax2.set_title(f"{titre} — Accuracy")
    ax2.set_xlabel("Epochs"); ax2.set_ylabel("Accuracy"); ax2.legend()

    plt.suptitle(f"{titre} — Courbes d'apprentissage", y=1.02, fontsize=12)
    plt.tight_layout()
    plt.show()

print("✅ Fonctions d'évaluation classification définies")

---
## 🧠 7. MLP 1 — 10 neurones (Classification)

In [ ]:
n_feat_clf = X_tr_clf_sc.shape[1]
es_clf = EarlyStopping(monitor="val_loss", patience=25, restore_best_weights=True, verbose=0)

# ── Modèle ───────────────────────────────────────────────────────────────────
mlp1_clf = Sequential([
    Dense(10, activation="relu", input_shape=(n_feat_clf,)),
    Dense(1,  activation="sigmoid")
], name="MLP1_Clf")
mlp1_clf.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
mlp1_clf.summary()

# ── Entraînement ─────────────────────────────────────────────────────────────
print("\nEntraînement MLP 1 (10 neurones)...")
h1_clf = mlp1_clf.fit(
    X_tr_clf_sc, y_tr_clf_np,
    epochs=400, batch_size=32,
    validation_split=0.15,
    callbacks=[es_clf], verbose=0
)
print(f"  ✅ Arrêt à l'epoch {len(h1_clf.history['loss'])} / 400")

plot_loss_clf(h1_clf, "MLP 1 (10 neurones)")

# ── Évaluation ───────────────────────────────────────────────────────────────
y_pred_mlp1_clf = (mlp1_clf.predict(X_te_clf_sc, verbose=0) > 0.5).astype(int).flatten()
acc_mlp1_clf = eval_clf(y_te_clf_np, y_pred_mlp1_clf, "MLP 1 — 10 neurones")

---
## 🧠 8. MLP 2 — 5 neurones (Classification)

In [ ]:
mlp2_clf = Sequential([
    Dense(5, activation="relu", input_shape=(n_feat_clf,)),
    Dense(1, activation="sigmoid")
], name="MLP2_Clf")
mlp2_clf.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

print("Entraînement MLP 2 (5 neurones)...")
h2_clf = mlp2_clf.fit(
    X_tr_clf_sc, y_tr_clf_np,
    epochs=400, batch_size=32,
    validation_split=0.15,
    callbacks=[es_clf], verbose=0
)
print(f"  ✅ Arrêt à l'epoch {len(h2_clf.history['loss'])} / 400")

plot_loss_clf(h2_clf, "MLP 2 (5 neurones)")

y_pred_mlp2_clf = (mlp2_clf.predict(X_te_clf_sc, verbose=0) > 0.5).astype(int).flatten()
acc_mlp2_clf = eval_clf(y_te_clf_np, y_pred_mlp2_clf, "MLP 2 — 5 neurones")

---
## ⚙️ 9. SVM — kernel RBF (Classification)

In [ ]:
print("Entraînement SVM (kernel RBF)...")
svm_clf = SVC(kernel="rbf", C=1.0, gamma="scale", random_state=42, probability=True)
svm_clf.fit(X_tr_clf_sc, y_tr_clf_np)
print("  ✅ SVM entraîné")

y_pred_svm_clf = svm_clf.predict(X_te_clf_sc)
acc_svm_clf = eval_clf(y_te_clf_np, y_pred_svm_clf, "SVM (kernel RBF)")

---
## 🌲 10. Random Forest (Classification)

In [ ]:
print("Entraînement Random Forest (100 arbres)...")
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_clf.fit(X_tr_clf, y_tr_clf_np)   # RF n'a pas besoin de normalisation
print("  ✅ Random Forest entraîné")

y_pred_rf_clf = rf_clf.predict(X_te_clf)
acc_rf_clf = eval_clf(y_te_clf_np, y_pred_rf_clf, "Random Forest (100 arbres)")

# ── Importance des features ───────────────────────────────────────────────────
feat_imp_clf = (pd.Series(rf_clf.feature_importances_, index=X_clf.columns)
                  .sort_values(ascending=True))

q75 = feat_imp_clf.quantile(0.75)
colors_imp = ["#c0392b" if v >= q75 else "#3498db" for v in feat_imp_clf]

plt.figure(figsize=(9, 9))
feat_imp_clf.plot(kind="barh", color=colors_imp, edgecolor="black", linewidth=0.5)
plt.axvline(feat_imp_clf.mean(), color="orange", ls="--", label=f"Moyenne = {feat_imp_clf.mean():.4f}")
plt.title("Importance des features — Random Forest (Classification)", fontsize=13)
plt.xlabel("Importance (Gini)")
plt.legend()
plt.tight_layout()
plt.show()

---
## 📈 11. Comparaison finale — Classification

In [ ]:
resultats_clf = {
    "MLP 1 (10 neurones)": acc_mlp1_clf,
    "MLP 2 (5 neurones)" : acc_mlp2_clf,
    "SVM (RBF)"          : acc_svm_clf,
    "Random Forest"      : acc_rf_clf,
}

print("\n" + "─" * 52)
print(f"  {'Modèle':<24}  {'Accuracy':>8}")
print("─" * 52)
best_acc = max(resultats_clf.values())
for nom, acc in sorted(resultats_clf.items(), key=lambda x: x[1], reverse=True):
    star = "  ⭐" if acc == best_acc else ""
    print(f"  {nom:<24}  {acc:>8.4f}{star}")
print("─" * 52)

# Graphique
fig, ax = plt.subplots(figsize=(9, 4))
noms_clf  = list(resultats_clf.keys())
accs_clf  = list(resultats_clf.values())
pal = ["#3498db", "#3498db", "#e67e22", "#27ae60"]

bars = ax.barh(noms_clf, accs_clf, color=pal, edgecolor="black", height=0.45)
ax.set_xlim(0, 1.15)
ax.axvline(0.5, color="gray", ls=":", alpha=0.6, label="Seuil 50%")
ax.set_xlabel("Accuracy")
ax.set_title("Comparaison des modèles — Classification (Passage de l'année)", fontsize=12)
for bar, val in zip(bars, accs_clf):
    ax.text(val + 0.008, bar.get_y() + bar.get_height() / 2,
            f"{val:.4f}", va="center", fontsize=10, fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

---
# 🟣 PARTIE 2 — Régression : Prédire la note finale G3

**Principe :** On prédit la **note finale G3** (0–20) en utilisant **toutes les données disponibles**,
y compris les notes intermédiaires G1 et G2.

> **Scénario réaliste :** en cours d'année scolaire, on connaît déjà les notes du 1er et 2ème trimestre.
> Peut-on prédire la note finale ?

**Métriques d'évaluation :**
- **MAE** — Erreur absolue moyenne (en points)
- **RMSE** — Racine de l'erreur quadratique moyenne
- **R²** — Coefficient de détermination (1.0 = parfait)
- **Accuracy exacte** — % de prédictions arrondies exactement correctes
- **Accuracy ±1 pt** — % de prédictions dans un rayon d'1 point


---
## 🔧 12. Prétraitement — Régression

In [ ]:
# ── Construction du jeu de données de régression ───────────────────────────
# Cible : G3 (note finale)  |  Features : tout sauf G3 (y compris G1, G2)
df_reg = data_enc.copy()

X_reg = df_reg.drop(columns=["G3"])
y_reg = df_reg["G3"]

print(f"Features : {X_reg.shape[1]}  |  Exemples : {X_reg.shape[0]}")
print(f"\nDistribution de G3 :")
print(f"  Min={y_reg.min()}  Max={y_reg.max()}  Moy={y_reg.mean():.2f}  Std={y_reg.std():.2f}")
print(f"  Médiane = {y_reg.median():.1f}  |  Élèves avec G3=0 : {(y_reg==0).sum()}")

# ── Histogramme de la distribution de G3 ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histogramme
axes[0].hist(y_reg, bins=range(0, 22), color="#8e44ad", edgecolor="black", alpha=0.85)
axes[0].axvline(x=10,          color="red",    ls="--", lw=1.5, label="Seuil passage (10)")
axes[0].axvline(x=y_reg.mean(), color="orange", ls="-.", lw=1.5, label=f"Moyenne = {y_reg.mean():.1f}")
axes[0].set_title("Distribution de la note finale G3", fontsize=12)
axes[0].set_xlabel("Note G3 (0–20)"); axes[0].set_ylabel("Nombre d'élèves")
axes[0].legend()

# Boxplot par mention
def note_vers_cat(n):
    if n < 10: return "Insuffisant (<10)"
    if n < 14: return "Passable (10–13)"
    if n < 17: return "Bien (14–16)"
    return "Très bien (17–20)"

df_reg_plot = df_reg.copy()
df_reg_plot["Mention"] = df_reg_plot["G3"].apply(note_vers_cat)
order_cat = ["Insuffisant (<10)", "Passable (10–13)", "Bien (14–16)", "Très bien (17–20)"]
pal_cat = {"Insuffisant (<10)": "#e74c3c", "Passable (10–13)": "#f39c12",
           "Bien (14–16)": "#2ecc71", "Très bien (17–20)": "#3498db"}
counts_cat = df_reg_plot["Mention"].value_counts().reindex(order_cat, fill_value=0)
axes[1].bar(order_cat, counts_cat.values,
            color=[pal_cat[c] for c in order_cat], edgecolor="black")
for i, v in enumerate(counts_cat.values):
    axes[1].text(i, v + 1, str(v), ha="center", fontsize=11, fontweight="bold")
axes[1].set_title("Répartition par mention", fontsize=12)
axes[1].set_ylabel("Nombre d'élèves")
axes[1].tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()

---
## 📊 13. Matrice de corrélation — Régression

In [ ]:
# ── Matrice de corrélation complète (avec G3) ────────────────────────────────
corr_reg = df_reg.corr()

plt.figure(figsize=(16, 14))
mask = np.triu(np.ones_like(corr_reg, dtype=bool))
sns.heatmap(corr_reg, mask=mask, annot=False, cmap="coolwarm", center=0,
            linewidths=0.3, linecolor="white",
            cbar_kws={"label": "Coefficient de Pearson", "shrink": 0.75})
plt.title("Matrice de corrélation — Données de régression", fontsize=14, pad=14)
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()

# ── Top corrélations avec G3 ─────────────────────────────────────────────────
print("\n🔍 Top 10 corrélations avec 'G3' :")
top_reg = (corr_reg["G3"]
           .drop("G3")
           .sort_values(key=lambda s: s.abs(), ascending=False)
           .head(10))
for feat, val in top_reg.items():
    sign = "+" if val > 0 else "-"
    bar = "█" * int(abs(val) * 30)
    print(f"  {sign}{abs(val):.4f}  {bar:<30}  {feat}")

# ── Focus : corrélations G1, G2, G3 entre eux ───────────────────────────────
notes_cols = ["G1", "G2", "G3"]
corr_notes = df_reg[notes_cols].corr()
plt.figure(figsize=(5, 4))
sns.heatmap(corr_notes, annot=True, fmt=".3f", cmap="YlOrRd",
            linewidths=2, linecolor="white", vmin=0, vmax=1)
plt.title("Corrélations entre notes G1, G2, G3", fontsize=12)
plt.tight_layout()
plt.show()

---
## ✂️ 14. Séparation train/test + normalisation — Régression

In [ ]:
X_tr_reg, X_te_reg, y_tr_reg, y_te_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

scaler_reg = StandardScaler()
X_tr_reg_sc = scaler_reg.fit_transform(X_tr_reg)
X_te_reg_sc  = scaler_reg.transform(X_te_reg)

y_tr_reg_np = y_tr_reg.values
y_te_reg_np = y_te_reg.values

print(f"Train : {X_tr_reg_sc.shape}  |  Test : {X_te_reg_sc.shape}")
print(f"G3 train — Moy={y_tr_reg_np.mean():.2f}  Std={y_tr_reg_np.std():.2f}")
print(f"G3 test  — Moy={y_te_reg_np.mean():.2f}  Std={y_te_reg_np.std():.2f}")

---
## 🛠️ 15. Fonctions d'évaluation — Régression

In [ ]:
# ── Labels des mentions pour la matrice de confusion ───────────────────────
MENTIONS_LABELS = [
    "Tres insuff (0-7)",
    "Insuffisant (8-9)",
    "Passable (10-11)",
    "Assez bien (12-13)",
    "Bien (14-15)",
    "Tres bien (16-20)"
]

def note_vers_mention(note):
    """Convertit une note (0-20) en mention textuelle."""    n = int(np.clip(round(float(note)), 0, 20))
    if n < 8:   return MENTIONS_LABELS[0]
    if n < 10:  return MENTIONS_LABELS[1]
    if n < 12:  return MENTIONS_LABELS[2]
    if n < 14:  return MENTIONS_LABELS[3]
    if n < 16:  return MENTIONS_LABELS[4]
    return MENTIONS_LABELS[5]


def eval_reg(y_true, y_pred, nom):
    """
    Évalue un modèle de régression :
    - MAE, RMSE, R²
    - Accuracy exacte + à ±1 point
    - Matrice de confusion par mention + scatter plot
    - Distribution des résidus
    """
    y_pred_clipped = np.clip(y_pred, 0, 20)

    mae  = mean_absolute_error(y_true, y_pred_clipped)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred_clipped))
    r2   = r2_score(y_true, y_pred_clipped)

    y_pred_int  = np.round(y_pred_clipped).astype(int)
    acc_exact   = accuracy_score(y_true.astype(int), y_pred_int)
    acc_1pt     = np.mean(np.abs(y_true - y_pred_clipped) <= 1.0)

    print(f"\n{'='*62}")
    print(f"  {nom}")
    print(f"{'─'*62}")
    print(f"  MAE    : {mae:.4f}  (erreur absolue moyenne en pts/20)")
    print(f"  RMSE   : {rmse:.4f}")
    print(f"  R²     : {r2:.4f}   (0=nul, 1=parfait)")
    print(f"{'─'*62}")
    print(f"  Accuracy exacte (arrondi)  : {acc_exact*100:.1f}%")
    print(f"  Accuracy ±1 point          : {acc_1pt*100:.1f}%")
    print(f"{'='*62}")

    # ── Matrice de confusion par mention ─────────────────────────────────────
    labs_true = [note_vers_mention(v) for v in y_true]
    labs_pred = [note_vers_mention(v) for v in y_pred_clipped]
    presents  = [m for m in MENTIONS_LABELS if m in labs_true or m in labs_pred]
    cm_reg    = confusion_matrix(labs_true, labs_pred, labels=presents)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Confusion matrix
    sns.heatmap(cm_reg, annot=True, fmt="d", cmap="Purples",
                xticklabels=presents, yticklabels=presents,
                ax=axes[0], linewidths=0.5)
    axes[0].set_xlabel("Prédit"); axes[0].set_ylabel("Réel")
    axes[0].set_title(f"Matrice de confusion (mentions)\n{nom}", fontsize=10)
    axes[0].tick_params(axis="x", rotation=35)
    axes[0].tick_params(axis="y", rotation=0)

    # Scatter plot prédictions vs réalité
    axes[1].scatter(y_true, y_pred_clipped, alpha=0.4, color="#8e44ad",
                    edgecolors="k", linewidths=0.2, s=30)
    axes[1].plot([0, 20], [0, 20], "r--", label="Parfait", lw=1.5)
    axes[1].fill_between([0, 20], [-1, 19], [1, 21],
                         alpha=0.12, color="green", label="±1 point")
    axes[1].set_xlim(-0.5, 20.5); axes[1].set_ylim(-0.5, 20.5)
    axes[1].set_xlabel("Note réelle G3"); axes[1].set_ylabel("Note prédite")
    axes[1].set_title(f"Prédictions vs Réalité\nAcc. ±1pt = {acc_1pt*100:.1f}%", fontsize=10)
    axes[1].legend(fontsize=9)

    # Distribution des résidus
    residus = y_true - y_pred_clipped
    axes[2].hist(residus, bins=25, color="#3498db", edgecolor="black", alpha=0.8)
    axes[2].axvline(0,             color="red",    ls="--", lw=1.5, label="Erreur nulle")
    axes[2].axvline(residus.mean(), color="orange", ls="-.", lw=1.5,
                    label=f"Moy = {residus.mean():.2f}")
    axes[2].set_title(f"Distribution des résidus\nMAE={mae:.3f}  RMSE={rmse:.3f}", fontsize=10)
    axes[2].set_xlabel("Erreur (réel − prédit)"); axes[2].set_ylabel("Fréquence")
    axes[2].legend(fontsize=9)

    plt.suptitle(nom, fontsize=12, y=1.01, fontweight="bold")
    plt.tight_layout()
    plt.show()

    return {"mae": mae, "rmse": rmse, "r2": r2,
            "acc_exact": acc_exact, "acc_1pt": acc_1pt}


def plot_loss_reg(history, titre):
    """Courbes d'apprentissage MSE + MAE pour un MLP de régression."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

    ax1.plot(history.history["loss"],     label="Train",      color="#8e44ad")
    ax1.plot(history.history["val_loss"], label="Validation",  color="#c0392b", ls="--")
    ax1.set_title(f"{titre} — Loss (MSE)")
    ax1.set_xlabel("Epochs"); ax1.set_ylabel("MSE"); ax1.legend()

    ax2.plot(history.history["mae"],     label="Train",      color="#16a085")
    ax2.plot(history.history["val_mae"], label="Validation",  color="#e67e22", ls="--")
    ax2.set_title(f"{titre} — MAE (points)")
    ax2.set_xlabel("Epochs"); ax2.set_ylabel("MAE"); ax2.legend()

    plt.suptitle(f"{titre} — Courbes d'apprentissage", y=1.02, fontsize=12)
    plt.tight_layout()
    plt.show()

print("✅ Fonctions d'évaluation régression définies")

---
## 🧠 16. MLP 1 — 10 neurones (Régression)

In [ ]:
n_feat_reg = X_tr_reg_sc.shape[1]
es_reg = EarlyStopping(monitor="val_loss", patience=30, restore_best_weights=True, verbose=0)

# ── Modèle ───────────────────────────────────────────────────────────────────
mlp1_reg = Sequential([
    Dense(10, activation="relu", input_shape=(n_feat_reg,)),
    Dense(1,  activation="linear")        # sortie linéaire pour la régression
], name="MLP1_Reg")
mlp1_reg.compile(optimizer="adam", loss="mse", metrics=["mae"])
mlp1_reg.summary()

# ── Entraînement ─────────────────────────────────────────────────────────────
print("\nEntraînement MLP 1 (10 neurones) — régression...")
h1_reg = mlp1_reg.fit(
    X_tr_reg_sc, y_tr_reg_np,
    epochs=500, batch_size=32,
    validation_split=0.15,
    callbacks=[es_reg], verbose=0
)
print(f"  ✅ Arrêt à l'epoch {len(h1_reg.history['loss'])} / 500")

plot_loss_reg(h1_reg, "MLP 1 (10 neurones)")

# ── Évaluation ───────────────────────────────────────────────────────────────
y_pred_mlp1_reg = mlp1_reg.predict(X_te_reg_sc, verbose=0).flatten()
res_mlp1_reg = eval_reg(y_te_reg_np, y_pred_mlp1_reg, "MLP 1 — 10 neurones (Régression)")

---
## 🧠 17. MLP 2 — 5 neurones (Régression)

In [ ]:
mlp2_reg = Sequential([
    Dense(5, activation="relu", input_shape=(n_feat_reg,)),
    Dense(1, activation="linear")
], name="MLP2_Reg")
mlp2_reg.compile(optimizer="adam", loss="mse", metrics=["mae"])

print("Entraînement MLP 2 (5 neurones) — régression...")
h2_reg = mlp2_reg.fit(
    X_tr_reg_sc, y_tr_reg_np,
    epochs=500, batch_size=32,
    validation_split=0.15,
    callbacks=[es_reg], verbose=0
)
print(f"  ✅ Arrêt à l'epoch {len(h2_reg.history['loss'])} / 500")

plot_loss_reg(h2_reg, "MLP 2 (5 neurones)")

y_pred_mlp2_reg = mlp2_reg.predict(X_te_reg_sc, verbose=0).flatten()
res_mlp2_reg = eval_reg(y_te_reg_np, y_pred_mlp2_reg, "MLP 2 — 5 neurones (Régression)")

---
## ⚙️ 18. SVR — kernel RBF (Régression)

In [ ]:
print("Entraînement SVR (kernel RBF, C=10, epsilon=0.5)...")
svr = SVR(kernel="rbf", C=10.0, epsilon=0.5, gamma="scale")
svr.fit(X_tr_reg_sc, y_tr_reg_np)
print("  ✅ SVR entraîné")

y_pred_svr = svr.predict(X_te_reg_sc)
res_svr = eval_reg(y_te_reg_np, y_pred_svr, "SVR (kernel RBF)")

---
## 🌲 19. Random Forest (Régression)

In [ ]:
print("Entraînement Random Forest Regressor (100 arbres)...")
rfr = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rfr.fit(X_tr_reg, y_tr_reg_np)
print("  ✅ Random Forest Regressor entraîné")

y_pred_rfr = rfr.predict(X_te_reg)
res_rfr = eval_reg(y_te_reg_np, y_pred_rfr, "Random Forest Regressor (100 arbres)")

# ── Importance des features ───────────────────────────────────────────────────
feat_imp_reg = (pd.Series(rfr.feature_importances_, index=X_reg.columns)
                  .sort_values(ascending=True))

q75_reg = feat_imp_reg.quantile(0.75)
colors_imp_reg = ["#c0392b" if v >= q75_reg else "#8e44ad" for v in feat_imp_reg]

plt.figure(figsize=(9, 9))
feat_imp_reg.plot(kind="barh", color=colors_imp_reg, edgecolor="black", linewidth=0.5)
plt.axvline(feat_imp_reg.mean(), color="orange", ls="--",
            label=f"Moyenne = {feat_imp_reg.mean():.4f}")
plt.title("Importance des features — Random Forest (Régression G3)", fontsize=13)
plt.xlabel("Importance")
plt.legend()
plt.tight_layout()
plt.show()

---
## 📈 20. Comparaison finale — Régression

In [ ]:
resultats_reg = {
    "MLP 1 (10 neurones)": res_mlp1_reg,
    "MLP 2 (5 neurones)" : res_mlp2_reg,
    "SVR (RBF)"          : res_svr,
    "Random Forest"      : res_rfr,
}

# ── Tableau récapitulatif ─────────────────────────────────────────────────────
best_r2 = max(v["r2"]  for v in resultats_reg.values())
best_mae = min(v["mae"] for v in resultats_reg.values())

print("\n" + "─"*72)
print(f"  {'Modèle':<22}  {'MAE':>6}  {'RMSE':>6}  {'R²':>6}  {'Acc.exacte':>10}  {'Acc.±1pt':>8}")
print("─"*72)
for nom, r in sorted(resultats_reg.items(), key=lambda x: x[1]["r2"], reverse=True):
    star = "  ⭐" if r["r2"] == best_r2 else ""
    print(f"  {nom:<22}  {r['mae']:>6.4f}  {r['rmse']:>6.4f}  "
          f"{r['r2']:>6.4f}  {r['acc_exact']:>10.4f}  {r['acc_1pt']:>8.4f}{star}")
print("─"*72)

# ── Graphiques comparatifs ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
noms_reg = list(resultats_reg.keys())
pal_reg  = ["#8e44ad", "#8e44ad", "#e67e22", "#27ae60"]

for ax, (metric, title, lower_better) in zip(axes, [
    ("mae",       "MAE (↓ mieux)",          True),
    ("r2",        "R² (↑ mieux)",           False),
    ("acc_1pt",   "Accuracy ±1 pt (↑ mieux)", False),
]):
    vals = [resultats_reg[n][metric] for n in noms_reg]
    bars = ax.bar(noms_reg, vals, color=pal_reg, edgecolor="black")
    ax.set_title(title, fontsize=11)
    ax.set_xticklabels(noms_reg, rotation=20, ha="right", fontsize=8)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() * 1.02, f"{v:.3f}",
                ha="center", fontsize=9, fontweight="bold")

plt.suptitle("Comparaison des modèles — Régression (Prédiction de G3)",
             fontsize=13, y=1.03, fontweight="bold")
plt.tight_layout()
plt.show()